<a href="https://colab.research.google.com/github/LuciaMellini/AMD_project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding similar items
The aim is to detect similar textual items in the `text` field of the Kaggle [Yelp](https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset) dataset.

In [1]:
import os
import json
import pandas as pd
import pip
import string

def import_or_install(package):
    try:
        __import__(package)
    except ImportError:
        pip.main(['install', package])

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [2]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

import findspark
findspark.init("spark-3.5.0-bin-hadoop3")# SPARK_HOME
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

sc = spark.sparkContext

We first of all import the Yelp dataset from Kaggle, using a token.

In [3]:
os.environ['KAGGLE_USERNAME'] = "xxx"
os.environ['KAGGLE_KEY'] = "xxx"
!kaggle datasets download -d yelp-dataset/yelp-dataset

100% 4.07G/4.07G [00:56<00:00, 134MB/s]
100% 4.07G/4.07G [00:56<00:00, 77.4MB/s]


In [4]:
from tqdm import tqdm
import zipfile

DATA_DIR = "/content/yelp-dataset"

with zipfile.ZipFile(DATA_DIR + ".zip","r") as zip_ref:
     for file in tqdm(iterable=zip_ref.namelist(), total=len(zip_ref.namelist())):
          zip_ref.extract(member=file, path=DATA_DIR)

100%|██████████| 6/6 [02:22<00:00, 23.71s/it]


We take into consideration the portion of the dataset regarding reviews, contained in `yelp_academic_dataset_review.json`. We convert the resulting dataframe in RDD form.

In [5]:
df_reviews = spark.read.json("/content/yelp-dataset/yelp_academic_dataset_review.json")
reviews_RDD = df_reviews.rdd

We sample the dataset at the only scope to speed up the following steps in the (free) Google Colab environment.

In [6]:
reviews_RDD = reviews_RDD.sample(False, 0.00001, 42)
reviews_num = reviews_RDD.count()

Seen the aim of the project we will only be looking at the `text` attribute of the imported reviews. From now on we will refer to the `text` field of a review, simply as *review*.

In [7]:
reviews_RDD = reviews_RDD.map((lambda r: (r[0], r['text'])))

Let's look at some reviews from the dataset.

In [8]:
print('---\n')
for text in reviews_RDD.take(3):
    print('{}\n'.format(text[1]))
    print('---\n')

---

This annual June event celebrating Indian jewelry, art and food turns out large crowds. First, the good: Lots of jewelry vendors offer tons of variety, from Santa Fe and New Mexico flavored pieces to bright, chunky tribal ones. Necklaces, door chains, bracelets, earrings, even beaded purses and more can be found here. One unusual piece that was just out of my price range (at $500!) was a gorgeous eagle carved out of bone or ivory on a circular metal necklace. 

Now the bad: The Eiteljorg's parking lot is not open to festivalgoers, despite the fact that they're the ones hosting the event and it's a $10 charge to enter. Further, the food offerings are more Hoosier than Hopi--the closest thing to Native American cuisine here was the beef jerky sold in the gift shop. I'm joking--a little. 

Indian Fry Tacos were sold, but they make use of flour and lard, ingredients introduced to Native Americans via the U.S. government following a forced expulsion during the Trail of Tears. One Mexic

## Data pre-processing

We begin by removing text cells that are `None` or that contain empty strings. It is possible to verify that for each review a nonempty text is present.

In [9]:
reviews_RDD = reviews_RDD.filter(lambda text: bool(text))

To study the similarity between reviews we proceed by looking at the relative string as a set of tokens. We have chosen to divide each review in the terms, all considered in lower case, that compose it. We have preferred this approach against using classical $k$-grams because we are more inteersted in the meaning of the reviews, so we don' care about looking also at the structure of the text.

We get rid of the stop words appearing in the tokens to extract the actual semantics of the text. Also we lemmatize the remaining tokens to consider as similar the various inflections of a certain word. Lastly, for each review we maintain each appearing token once.

In [10]:
%%capture
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('english')
import spacy
nlp = spacy.load("en_core_web_sm")

split_regex = r'\W+'

tokenize = lambda string: [s for s in re.split(split_regex,string.lower()) if s not in stopwords and s != '']
review_tokens_RDD = reviews_RDD.map(lambda s: (s[0],tokenize(s[1])))

lemmatize = lambda word: str([token.lemma_ for token in nlp(word)][0])
tokens_in_review_RDD = review_tokens_RDD.map(lambda s: (s[0], [lemmatize(w) for w in s[1]]))

tokens_in_review_RDD = tokens_in_review_RDD.map(lambda s: (s[0], list(set(s[1]))))

Let's look at how the first of the reviews showed above has been transformed.

In [11]:
tokens_in_review_RDD.first()

('4ieBZ94-Bpn7BBvN1fGCyQ',
 ['shop',
  'little',
  'ice',
  'vendor',
  'find',
  'charge',
  'eiteljorg',
  'enter',
  'state',
  'cuisine',
  'flour',
  'tamale',
  'joke',
  'chunky',
  'price',
  'even',
  'lot',
  'door',
  'mexico',
  'turn',
  'parking',
  'open',
  'ton',
  'chain',
  'authentic',
  'new',
  'close',
  'cream',
  'delicious',
  'bead',
  'bright',
  'bone',
  'one',
  'range',
  'force',
  'bad',
  'make',
  'festivalgoer',
  'variety',
  'offering',
  'fry',
  'unusual',
  'jewelry',
  'ingredient',
  'regular',
  'gorgeous',
  'hope',
  'use',
  'purse',
  'american',
  'tacos',
  'via',
  'eagle',
  'root',
  'june',
  'hopi',
  'beef',
  'large',
  'first',
  'crowd',
  'americans',
  'follow',
  'jerky',
  'carve',
  '10',
  'gift',
  'trail',
  'piece',
  'indian',
  'u',
  'native',
  'flavor',
  'hoosier',
  'tear',
  '500',
  'fe',
  'tribal',
  'fact',
  'celebrate',
  'art',
  'thing',
  'despite',
  'food',
  'expulsion',
  'santa',
  'government',


At this point we have codified each review through it's essential information.

## Similar items with Jaccard similarity

We begin by evaluating the similarity of the reviews according to the Jaccard similarity measure between sets. The Jaccard similarity between sets $S$ and $T$ is defined as:
\begin{equation}
J(S,T)=\frac{|S\cap T|}{|S\cup T|}
\end{equation}
In our case the sets have as items the tokens that we have extracted from the reviews.



### Similarity preserving summary of the reviews
To hold the summary of the tokens extracted from each review compactly we consider their characteristic matrix, used to represent a collection of sets.

Though with growing amounts of data it is not realistic to maintain the whole characteristic matrix. So, we construct a succint structure called a signature matrix. Each row of such matrix is constructed as follows:
1. choose a permutation of the rows of the characteristic matrix uniformly at random among all possible permutations,
2. apply the chosen permutation to the rows of the charactistic matrix,
3. apply a minhash function to all columns of the resulting matrix.

A minhash function is defined as $h:\{\text{reviews}\}\to\{\text{shingles}\}$, and for a set of reviews it returns the index of the first one for which a certain token is present.

The column related to a review is it's signature.

We begin by putting aside all the *distinct* tokens found in the dataset, and their amount.

In [25]:
all_tokens_RDD = tokens_in_review_RDD.flatMap(lambda t: [(s, 1) for s in t[1]]).reduceByKey(lambda a,b: a+b).map(lambda s: (s[0],1))
all_tokens_num = all_tokens_RDD.count()

We compute the minhash for each review, so we are looking at the columns of the signature matrix.

All the found tokens are permuted by assigning them to the index of their new position. The minhash of a given review is the position of the first $1$ that appears in the signature matrix that has undergone this permutation. This value is given by the minimum index among the tokens appearing in such review.

In [34]:
import numpy as np
signature_length = 42
permutations_RDD = sc.parallelize(range(signature_length)).map(lambda s: (s,list(np.random.permutation(all_tokens_num))))
all_tokens_index_RDD = all_tokens_RDD.map(lambda s: s[0]).zipWithIndex()


In [39]:
token_permutations_RDD = all_tokens_index_RDD.cartesian(permutations_RDD).map(lambda s: (s[0][0],(s[1][0],s[1][1][s[0][1]]))).groupByKey().mapValues(list)
reviews_for_token_RDD = tokens_in_review_RDD.flatMap(lambda s: [(t,s[0]) for t in s[1]]).groupByKey().mapValues(list)
token_review_indices_RDD = token_permutations_RDD.join(reviews_for_token_RDD)

token_index_review_RDD = token_review_indices_RDD.map(lambda s: ((s[0],s[1][0]),s[1][1])).flatMapValues(lambda s: s)
review_indices_RDD = token_index_review_RDD.map(lambda s: (s[1],s[0][1])).flatMapValues(lambda s: s)
review_minhash_RDD = review_indices_RDD.map(lambda s: ((s[0],s[1][0]),s[1][1])).groupByKey().mapValues(list).map(lambda s: (s[0],min(s[1])))
review_signature_RDD = review_minhash_RDD.map(lambda s: (s[0][0],s[1])).groupByKey().mapValues(list)

We can use the the signature matrix to estimate the Jaccard similarity between reviews. It is provable that $\mathbb{P}(H(S_1)=H(S_2))=J(S_1,S_2)$, where $H$ is a minhash function applied to a random permutation of the review set. So, by estimating the probability with the relative frequency of having the same signature for some pair of reviews, we obtain their approximate similarity.

### Locality-sensitive hashing (LSH)

Considering the number of rows in the signature matrix it would be too costly to scan all of them to compute the relative frequency between all possible pairs of reviews. So, we proceed by applying Locality-Sensitive Hashing. In this approach we filter on pairs of reviews by hashing them several times and only looking at those couples collected in the same bucket. The rationale is that similar reviews are more likely to be hashed in the same bucket, so we hope that dissimilar pairs end up in distinct buckets, and thus are never checked for similarity.